In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

In [2]:
import pandas as pd

from quant_ml.config import RAW_DATA_DIR, INTERIM_DATA_DIR, START_DATE, END_DATE
from quant_ml.features import (
    reshape_ohlcv_to_long,
    add_return_features,
    add_trend_features,
    add_volatility_features,
    add_volume_features,
    prepare_macro_features,
    merge_price_and_macro,
)
from quant_ml.labels import add_direction_label

In [3]:
prices = pd.read_parquet(RAW_DATA_DIR / "yahoo_prices.parquet")
macro = pd.read_parquet(RAW_DATA_DIR / "fred_macro.parquet")

In [4]:
price_long = reshape_ohlcv_to_long(prices)
price_long.head()

,date,ticker,open,high,low,close,volume
0,2012-01-03,EEM,28.671159,28.884465,28.604961,28.759422,72557500
1,2012-01-04,EEM,28.494634,28.663807,28.391657,28.597609,46225600
2,2012-01-05,EEM,28.421072,28.560825,28.193058,28.472559,54383000
3,2012-01-06,EEM,28.443141,28.443141,28.097441,28.112152,50333100
4,2012-01-09,EEM,28.376945,28.465209,28.222481,28.406364,45814800


In [5]:
print(price_long.shape)
print(price_long.columns)

(21114, 7)
Index(['date', 'ticker', 'open', 'high', 'low', 'close', 'volume'], dtype='object')


In [6]:
macro_prepared = prepare_macro_features(
    macro=macro,
    start_date=START_DATE,
    end_date=END_DATE,
)

macro_prepared.head()

,fed_funds,cpi,unemployment,10y_treasury,2y_treasury,term_spread
date,,,,,,
2012-01-01,0.08,227.842,8.3,1.97,0.24,1.73
2012-02-01,0.10,228.329,8.3,1.97,0.28,1.69
2012-03-01,0.13,228.807,8.2,2.17,0.34,1.83
2012-04-01,0.14,229.187,8.2,2.05,0.29,1.76
2012-05-01,0.16,228.713,8.2,1.80,0.29,1.51


In [7]:
model_df = merge_price_and_macro(price_long, macro_prepared)
model_df.head()

,date,ticker,open,high,low,close,volume,fed_funds,cpi,unemployment,10y_treasury,2y_treasury,term_spread
0,2012-01-03,EEM,28.671159,28.884465,28.604961,28.759422,72557500,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-01-04,EEM,28.494634,28.663807,28.391657,28.597609,46225600,NaN,NaN,NaN,NaN,NaN,NaN
2,2012-01-05,EEM,28.421072,28.560825,28.193058,28.472559,54383000,NaN,NaN,NaN,NaN,NaN,NaN
3,2012-01-06,EEM,28.443141,28.443141,28.097441,28.112152,50333100,NaN,NaN,NaN,NaN,NaN,NaN
4,2012-01-09,EEM,28.376945,28.465209,28.222481,28.406364,45814800,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
model_df = add_return_features(model_df)
model_df = add_trend_features(model_df)
model_df = add_volatility_features(model_df)
model_df = add_volume_features(model_df)
model_df = add_direction_label(model_df, horizon=5)

model_df.head(15)

,date,ticker,open,high,low,close,volume,fed_funds,cpi,unemployment,...,ma_ratio_20,ma_ratio_50,ma_ratio_100,ma_cross_20_50,vol_20d,vol_60d,volume_z_20,volume_ratio_20,fwd_ret_5d,target_up_5d
0,2012-01-03,EEM,28.671159,28.884465,28.604961,28.759422,72557500,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.009207,1.0
1,2012-01-04,EEM,28.494634,28.663807,28.391657,28.597609,46225600,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.015689,1.0
2,2012-01-05,EEM,28.421072,28.560825,28.193058,28.472559,54383000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.024283,1.0
3,2012-01-06,EEM,28.443141,28.443141,28.097441,28.112152,50333100,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.027996,1.0
4,2012-01-09,EEM,28.376945,28.465209,28.222481,28.406364,45814800,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.036769,1.0
5,2012-01-10,EEM,29.053636,29.186032,28.950661,29.024214,59620000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.040294,1.0
6,2012-01-11,EEM,28.855039,29.060988,28.774130,29.046280,47747100,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.048620,1.0
7,2012-01-12,EEM,29.193388,29.237521,28.965374,29.163969,43328400,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.043632,1.0
8,2012-01-13,EEM,28.950665,28.965376,28.641739,28.899178,57195200,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.063374,1.0
9,2012-01-17,EEM,29.524379,29.605288,29.333141,29.450827,70477600,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.044955,1.0


In [9]:
model_df.isna().sum().sort_values(ascending=False).head(20)

ma_ratio_100       594
ret_60d            360
vol_60d            360
ma_ratio_50        294
ma_cross_20_50     294
ret_20d            120
term_spread        120
2y_treasury        120
10y_treasury       120
unemployment       120
cpi                120
fed_funds          120
vol_20d            120
volume_z_20        114
volume_ratio_20    114
ma_ratio_20        114
ret_5d              30
fwd_ret_5d          30
ret_1d               6
date                 0
dtype: int64

In [10]:
feature_cols = [
    "ret_1d", "ret_5d", "ret_20d", "ret_60d",
    "ma_ratio_20", "ma_ratio_50", "ma_ratio_100", "ma_cross_20_50",
    "vol_20d", "vol_60d",
    "volume_z_20", "volume_ratio_20",
    "fed_funds", "cpi", "unemployment", "10y_treasury", "2y_treasury", "term_spread",
]

target_col = "target_up_5d"
fwd_ret_col = "fwd_ret_5d"

final_df = model_df.dropna(subset=feature_cols + [target_col, fwd_ret_col]).copy()

print(final_df.shape)
final_df.head()

(20490, 27)


,date,ticker,open,high,low,close,volume,fed_funds,cpi,unemployment,...,ma_ratio_20,ma_ratio_50,ma_ratio_100,ma_cross_20_50,vol_20d,vol_60d,volume_z_20,volume_ratio_20,fwd_ret_5d,target_up_5d
99,2012-05-24,EEM,27.678185,27.685539,27.192732,27.457525,57707700,0.16,228.713,8.2,...,-0.064118,-0.101638,-0.110964,-0.040091,0.010547,0.012934,-0.023332,0.993897,-0.017145,0.0
100,2012-05-25,EEM,27.347199,27.486950,27.251579,27.325134,47685000,0.16,228.713,8.2,...,-0.062567,-0.102952,-0.114840,-0.043080,0.010403,0.012836,-0.726949,0.815904,-0.008076,0.0
101,2012-05-29,EEM,28.060670,28.215132,27.854718,28.097446,71172100,0.16,228.713,8.2,...,-0.031157,-0.074966,-0.089674,-0.045218,0.012968,0.013456,0.776170,1.185941,-0.032723,0.0
102,2012-05-30,EEM,27.560501,27.692897,27.435458,27.611988,80092500,0.16,228.713,8.2,...,-0.041870,-0.088131,-0.105153,-0.048282,0.012950,0.013457,1.279838,1.289009,0.013053,1.0
103,2012-05-31,EEM,27.663479,27.869428,27.376621,27.729677,66528900,0.16,228.713,8.2,...,-0.031956,-0.081809,-0.101227,-0.051499,0.013127,0.012851,0.225904,1.044534,0.014854,1.0


In [11]:
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [12]:
final_df.to_parquet(INTERIM_DATA_DIR / "model_dataset.parquet", index=False)

In [13]:
final_df[target_col].value_counts(normalize=True)

target_up_5d
1.0    0.56408
0.0    0.43592
Name: proportion, dtype: float64

In [17]:
print(
    price_long.shape,
    final_df.shape,
)
final_df.head()

(21114, 7) (20490, 27)


,date,ticker,open,high,low,close,volume,fed_funds,cpi,unemployment,...,ma_ratio_20,ma_ratio_50,ma_ratio_100,ma_cross_20_50,vol_20d,vol_60d,volume_z_20,volume_ratio_20,fwd_ret_5d,target_up_5d
99,2012-05-24,EEM,27.678185,27.685539,27.192732,27.457525,57707700,0.16,228.713,8.2,...,-0.064118,-0.101638,-0.110964,-0.040091,0.010547,0.012934,-0.023332,0.993897,-0.017145,0.0
100,2012-05-25,EEM,27.347199,27.486950,27.251579,27.325134,47685000,0.16,228.713,8.2,...,-0.062567,-0.102952,-0.114840,-0.043080,0.010403,0.012836,-0.726949,0.815904,-0.008076,0.0
101,2012-05-29,EEM,28.060670,28.215132,27.854718,28.097446,71172100,0.16,228.713,8.2,...,-0.031157,-0.074966,-0.089674,-0.045218,0.012968,0.013456,0.776170,1.185941,-0.032723,0.0
102,2012-05-30,EEM,27.560501,27.692897,27.435458,27.611988,80092500,0.16,228.713,8.2,...,-0.041870,-0.088131,-0.105153,-0.048282,0.012950,0.013457,1.279838,1.289009,0.013053,1.0
103,2012-05-31,EEM,27.663479,27.869428,27.376621,27.729677,66528900,0.16,228.713,8.2,...,-0.031956,-0.081809,-0.101227,-0.051499,0.013127,0.012851,0.225904,1.044534,0.014854,1.0


In [15]:
final_df[target_col].value_counts(normalize=True)

target_up_5d
1.0    0.56408
0.0    0.43592
Name: proportion, dtype: float64

In [ ]:
final_df.to_parquet(
    INTERIM_DATA_DIR / "model_dataset_v1.parquet",
    index=False
)